# 01a0 — Blockchain & smart contracts primer

A ground-up, hands-on tour. We drive a real local Ethereum chain (`anvil`) through `cast` and `forge`, naming each concept after we've executed it. By the last section you'll be able to read [BandwidthEscrow.sol](../../contracts/src/BandwidthEscrow.sol) line by line.

**Prereq:** `anvil`, `cast`, `forge` on PATH (Foundry installed). Run cells top to bottom — anvil is started in the next cell and killed in the very last cell.

In [1]:
# --- Notebook runtime setup ---------------------------------------
import atexit, subprocess, time, shutil, sys, pathlib, json, os

PRIMER_DIR = pathlib.Path.cwd().resolve()
REPO_ROOT = PRIMER_DIR.parent.parent
RPC = 'http://127.0.0.1:8545'

def run(cmd, cwd=None, check=True):
    """Run a shell command, show it, return stdout."""
    print('$', ' '.join(str(c) for c in cmd))
    r = subprocess.run(cmd, cwd=cwd or PRIMER_DIR, capture_output=True, text=True)
    if r.stdout: print(r.stdout.rstrip())
    if r.returncode != 0:
        if r.stderr: print(r.stderr.rstrip(), file=sys.stderr)
        if check: raise SystemExit(f'command failed: {cmd}')
    return r.stdout.strip()

for tool in ('anvil', 'cast', 'forge'):
    assert shutil.which(tool), f'{tool} not found on PATH'
print('Foundry tools OK')

Foundry tools OK


In [2]:
# --- Start anvil --------------------------------------------------
_anvil_proc = subprocess.Popen(
    ['anvil', '--host', '127.0.0.1', '--port', '8545', '--silent'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
atexit.register(_anvil_proc.terminate)

# Wait for RPC to respond.
for _ in range(30):
    try:
        run(['cast', 'block-number', '--rpc-url', RPC], check=True)
        break
    except SystemExit:
        time.sleep(0.2)
else:
    raise RuntimeError('anvil did not come up')
print(f'anvil PID={_anvil_proc.pid}')

$ cast block-number --rpc-url http://127.0.0.1:8545


Error: error sending request for url (http://127.0.0.1:8545/)

Context:
- Error #0: client error (Connect)
- Error #1: tcp connect error
- Error #2: Connection refused (os error 111)


$ cast block-number --rpc-url http://127.0.0.1:8545
0
anvil PID=157117


## 1. What is a chain, really

You already know the intuition: a blockchain is an append-only distributed database of transactions. Let's make that concrete.

We started `anvil` — a process that pretends to be the entire Ethereum network. One node, no peers, no proof-of-stake. It exposes the same JSON-RPC interface mainnet does, on `http://127.0.0.1:8545`.

The chain has two things we'll keep separate in our heads:

1. **State** — current balances and contract storage (the "database").
2. **History** — the ordered list of blocks, each containing the    transactions that produced the next state.

Let's poke at both.

In [3]:
block_number = run(['cast', 'block-number', '--rpc-url', RPC])
print(f'\ncurrent block number: {block_number}')

$ cast block-number --rpc-url http://127.0.0.1:8545
0

current block number: 0


In [4]:
# The block itself. Block 0 is the genesis block — empty, no parent.
run(['cast', 'block', '0', '--rpc-url', RPC])

$ cast block 0 --rpc-url http://127.0.0.1:8545


baseFeePerGas        1000000000
difficulty           0
extraData            0x
gasLimit             30000000
gasUsed              0
hash                 0xbc29b21632240561c2af4a23e2dbc5bf7481c43e2b00ede0ce6b7afcc7314538
logsBloom            0x00000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
miner                0x0000000000000000000000000000000000000000
mixHash              0x0000000000000000000000000000000000000000000000000000000000000000
nonce                0x0000000000000000
num

'baseFeePerGas        1000000000\ndifficulty           0\nextraData            0x\ngasLimit             30000000\ngasUsed              0\nhash                 0xbc29b21632240561c2af4a23e2dbc5bf7481c43e2b00ede0ce6b7afcc7314538\nlogsBloom            0x00000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000\nminer                0x0000000000000000000000000000000000000000\nmixHash              0x0000000000000000000000000000000000000000000000000000000000000000\nnonce                0x0000000000000000\nnumber               0\nparentHash       

Notice the fields: `number`, `timestamp`, `parentHash`, `stateRoot`, `transactionsRoot`. Each block points to its parent by hash — that's the "chain" part. The `stateRoot` is a Merkle root summarising the entire state at this block — change one balance, the root changes, and so does the block hash.

## 2. Accounts & keys

An Ethereum **account** is just a keypair. Anvil generates 10 funded accounts deterministically on startup — same mnemonic every time, so the addresses and private keys are stable across runs.

We'll use these two throughout:

| Role | Address | Private key |
|---|---|---|
| Alice (acct 0) | `0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266` | `0xac0974...ff80` |
| Bob (acct 1) | `0x70997970C51812dc3A010C7d01b50e0d17dc79C8` | `0x59c699...690d` |

(Full keys are below — these are well-known test keys. **Never use them on a real network.**)

In [5]:
ALICE = '0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266'
ALICE_PK = '0xac0974bec39a17e36ba4a6b4d238ff944bacb478cbed5efcae784d7bf4f2ff80'
BOB   = '0x70997970C51812dc3A010C7d01b50e0d17dc79C8'
BOB_PK = '0x59c6995e998f97a5a0044966f0945389dc9e86dae88c7a8412f4603b6b78690d'

# Derive Alice's address from her private key to prove the link.
derived = run(['cast', 'wallet', 'address', ALICE_PK])
assert derived.lower().endswith(ALICE[2:].lower()), (derived, ALICE)
print('derived address matches Alice')

$ cast wallet address 0xac0974bec39a17e36ba4a6b4d238ff944bacb478cbed5efcae784d7bf4f2ff80
0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266
derived address matches Alice


**How the address is derived:** take the public key, hash it with `keccak256`, keep the last 20 bytes. That's it. No central registry, no certificate authority. Whoever holds the private key controls the account because only they can produce signatures that verify against the public key.

The private key never leaves the holder. Signing happens locally; only the signature goes on-chain.

## 3. A transaction, the smallest unit

A transaction is the only way to change state. Even deploying a contract or running a function is just "send a tx."

We're going to send 1 ETH from Alice to Bob. Three things happen:

1. Alice signs the tx **locally** with her private key.
2. The signed bytes are sent to anvil over JSON-RPC.
3. anvil includes the tx in a block. The tx now exists forever; the    state (balances) is updated accordingly.

In [6]:
# Send 1 ETH from Alice to Bob. --private-key tells cast which key to sign with.
tx_hash = run([
    'cast', 'send', BOB, '--value', '1ether',
    '--private-key', ALICE_PK, '--rpc-url', RPC,
    '--json',
])
import json
receipt = json.loads(tx_hash)
tx_hash = receipt['transactionHash']
print(f'\ntx hash: {tx_hash}')

$ cast send 0x70997970C51812dc3A010C7d01b50e0d17dc79C8 --value 1ether --private-key 0xac0974bec39a17e36ba4a6b4d238ff944bacb478cbed5efcae784d7bf4f2ff80 --rpc-url http://127.0.0.1:8545 --json
{"status":"0x1","cumulativeGasUsed":"0x5208","logs":[],"logsBloom":"0x00000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000","type":"0x2","transactionHash":"0xce09985d94caad02fc26f64a68a95d3ae7ec58fa0ef5208c672d4f42052488dc","transactionIndex":"0x0","blockHash":"0xcda56baec67a27fff6dd7a39a89cc855eb1641edad3e751eeb05bdf2386306a9","blockNumber":"0x1",

The **transaction hash** is `keccak256(rlp(signed_tx))` — a content-addressed fingerprint. Changing any field of the tx changes the hash; that's how the network refers to txs without trusting any label.

Let's pull the tx itself, then its receipt.

In [7]:
run(['cast', 'tx', tx_hash, '--rpc-url', RPC])

$ cast tx 0xce09985d94caad02fc26f64a68a95d3ae7ec58fa0ef5208c672d4f42052488dc --rpc-url http://127.0.0.1:8545

blockHash            0xcda56baec67a27fff6dd7a39a89cc855eb1641edad3e751eeb05bdf2386306a9
blockNumber          1
from                 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266
transactionIndex     0
effectiveGasPrice    1000000001

accessList           []
chainId              31337
gasLimit             21000
hash                 0xce09985d94caad02fc26f64a68a95d3ae7ec58fa0ef5208c672d4f42052488dc
input                0x
maxFeePerGas         2000000001
maxPriorityFeePerGas 1
nonce                0
r                    0xa243aa6e2c9c556166678a51151b1b1337f21443dc655f146d58187c2fdbd80e
s                    0x3df633138a48495a65e6230e1029da2340be19fb5e8f7f5031601e4ea8e6778c
to                   0x70997970C51812dc3A010C7d01b50e0d17dc79C8
type                 2
value                1000000000000000000
yParity              1


'blockHash            0xcda56baec67a27fff6dd7a39a89cc855eb1641edad3e751eeb05bdf2386306a9\nblockNumber          1\nfrom                 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266\ntransactionIndex     0\neffectiveGasPrice    1000000001\n\naccessList           []\nchainId              31337\ngasLimit             21000\nhash                 0xce09985d94caad02fc26f64a68a95d3ae7ec58fa0ef5208c672d4f42052488dc\ninput                0x\nmaxFeePerGas         2000000001\nmaxPriorityFeePerGas 1\nnonce                0\nr                    0xa243aa6e2c9c556166678a51151b1b1337f21443dc655f146d58187c2fdbd80e\ns                    0x3df633138a48495a65e6230e1029da2340be19fb5e8f7f5031601e4ea8e6778c\nto                   0x70997970C51812dc3A010C7d01b50e0d17dc79C8\ntype                 2\nvalue                1000000000000000000\nyParity              1'

Field-by-field:

- **`from`** — recovered from the signature `(v, r, s)`, not sent explicitly.
- **`to`** — Bob's address. If empty, this would be a contract deployment.
- **`value`** — wei being transferred (1 ETH = 10¹⁸ wei).
- **`nonce`** — counter per sender; prevents replay. Alice's first tx is nonce 0.
- **`gas`, `gasPrice`** — the fee budget.
- **`input`** — empty for a plain transfer; we'll see it filled later.
- **`v`, `r`, `s`** — the ECDSA signature.

The **receipt** is the post-execution summary.

In [8]:
run(['cast', 'receipt', tx_hash, '--rpc-url', RPC])

$ cast receipt 0xce09985d94caad02fc26f64a68a95d3ae7ec58fa0ef5208c672d4f42052488dc --rpc-url http://127.0.0.1:8545

blockHash            0xcda56baec67a27fff6dd7a39a89cc855eb1641edad3e751eeb05bdf2386306a9
blockNumber          1
contractAddress      
cumulativeGasUsed    21000
effectiveGasPrice    1000000001
from                 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266
gasUsed              21000
logs                 []
logsBloom            0x00000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
root                 
status              

'blockHash            0xcda56baec67a27fff6dd7a39a89cc855eb1641edad3e751eeb05bdf2386306a9\nblockNumber          1\ncontractAddress      \ncumulativeGasUsed    21000\neffectiveGasPrice    1000000001\nfrom                 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266\ngasUsed              21000\nlogs                 []\nlogsBloom            0x00000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000\nroot                 \nstatus               1 (success)\ntransactionHash      0xce09985d94caad02fc26f64a68a95d3ae7ec58fa0ef5208c672d4f42052488dc\nt

`status` `1` means success. `gasUsed` is what Alice actually paid for in computational work. `logs` is empty here (a transfer emits none) — we'll see logs in §8 when we discuss events.

## 4. State vs history

Two clocks tick in parallel:

- **State** — the current snapshot. "Alice has X wei." Mutable.   Read via `cast balance`, `cast storage`, contract view calls.
- **History** — the chain of blocks, each containing the txs that produced   the next state. Immutable. Read via `cast block`, `cast tx`, `cast receipt`.

The state is *derived* from the history: replay every tx from genesis and you get the current state. That's why the chain is auditable.

In [9]:
# Current balances
alice_bal = run(['cast', 'balance', ALICE, '--rpc-url', RPC])
bob_bal   = run(['cast', 'balance', BOB,   '--rpc-url', RPC])
print(f'\nAlice: {alice_bal} wei')
print(f'Bob:   {bob_bal} wei')

$ cast balance 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266 --rpc-url http://127.0.0.1:8545
9998999978999999979000
$ cast balance 0x70997970C51812dc3A010C7d01b50e0d17dc79C8 --rpc-url http://127.0.0.1:8545
10001000000000000000000

Alice: 9998999978999999979000 wei
Bob:   10001000000000000000000 wei


In [10]:
# Same question, but as of block 0 (before the transfer in §3).
alice_bal0 = run(['cast', 'balance', ALICE, '--block', '0', '--rpc-url', RPC])
bob_bal0   = run(['cast', 'balance', BOB,   '--block', '0', '--rpc-url', RPC])
print(f'\nAlice @ block 0: {alice_bal0} wei')
print(f'Bob   @ block 0: {bob_bal0} wei')

$ cast balance 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266 --block 0 --rpc-url http://127.0.0.1:8545
10000000000000000000000
$ cast balance 0x70997970C51812dc3A010C7d01b50e0d17dc79C8 --block 0 --rpc-url http://127.0.0.1:8545
10000000000000000000000

Alice @ block 0: 10000000000000000000000 wei
Bob   @ block 0: 10000000000000000000000 wei


Notice: the *historical* balances differ from the current ones. That's the point — the chain remembers every state it ever held, because it remembers every tx.

(Anvil keeps full archive state by default. Real Ethereum nodes can drop historical state to save disk, but the txs themselves never go away.)

## 5. From transactions to code: deploying `Counter.sol`

Until now, our txs only moved ETH. The next leap: a tx can also **deploy code**. A *smart contract* is just an account whose `code` field is non-empty. When you send a tx to it, the EVM runs that code.

We'll deploy this tiny contract (see `contracts/Counter.sol`):

```solidity
contract Counter {
    uint256 public number;
    function increment() external { number += 1; }
    function incrementBounded() external {
        require(number < 5, "max reached");
        number += 1;
    }
}
```

First, compile it with `forge build`.

In [11]:
run(['forge', 'build'])

$ forge build
No files changed, compilation skipped


'No files changed, compilation skipped'

In [12]:
# Deploy. `forge create` sends a creation tx (no `to`, code in `input`).
output = run([
    'forge', 'create', 'contracts/Counter.sol:Counter',
    '--private-key', ALICE_PK, '--rpc-url', RPC,
    '--broadcast',
])
# Extract the deployed address from forge's output.
import re
m = re.search(r'Deployed to:\s*(0x[0-9a-fA-F]{40})', output)
assert m, output
COUNTER = m.group(1)
print(f'\nCounter deployed at: {COUNTER}')

$ forge create contracts/Counter.sol:Counter --private-key 0xac0974bec39a17e36ba4a6b4d238ff944bacb478cbed5efcae784d7bf4f2ff80 --rpc-url http://127.0.0.1:8545 --broadcast
No files changed, compilation skipped
Deployer: 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266
Deployed to: 0xe7f1725E7734CE288F8367e1Bb143E90bb3F0512
Transaction hash: 0xbe9198b7383f3ae0e8f43009fbc1ba8d9438e437a533ff0fda2380c2d777383f

Counter deployed at: 0xe7f1725E7734CE288F8367e1Bb143E90bb3F0512


In [13]:
# Confirm: the deployed account has CODE. A regular EOA does not.
counter_code = run(['cast', 'code', COUNTER, '--rpc-url', RPC])
alice_code   = run(['cast', 'code', ALICE,   '--rpc-url', RPC])
print(f'\nCounter code length: {len(counter_code)} chars')
print(f'Alice code:          {alice_code!r}  (empty)')

$ cast code 0xe7f1725E7734CE288F8367e1Bb143E90bb3F0512 --rpc-url http://127.0.0.1:8545
0x6080604052348015600e575f80fd5b5060043610603a575f3560e01c806326e1dba314603e5780638381f58a146046578063d09de08a14605f575b5f80fd5b60446065565b005b604d5f5481565b60405190815260200160405180910390f35b604460a7565b60055f541060a75760405162461bcd60e51b815260206004820152600b60248201526a1b585e081c995858da195960aa1b604482015260640160405180910390fd5b60015f8082825460b6919060bd565b9091555050565b8082018082111560db57634e487b7160e01b5f52601160045260245ffd5b9291505056fea264697066735822122023d1d04088000fbf7fe2e0efeba116c3a765f6466bdb5ff6597b5ce9a407d95764736f6c63430008140033
$ cast code 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266 --rpc-url http://127.0.0.1:8545
0x

Counter code length: 560 chars
Alice code:          '0x'  (empty)


That's it. **A smart contract is an account with code.** The code is EVM bytecode — a stack-machine instruction stream. When the network sees a tx whose `to` is this address, every node runs the bytecode against the input, applies the resulting state changes, and agrees on the outcome (because the EVM is deterministic).

## 6. EVM execution: `send` vs `call`, gas, revert

Two ways to interact with a deployed contract:

- **`cast send`** — sends a real, signed tx. State changes. Costs gas.   Mined into a block.
- **`cast call`** — a *local simulation*. The node runs the function   against current state but discards the result. No tx, no block, no   gas paid. Used for reading view functions or previewing a call's   return value.

Let's increment, then read.

In [14]:
run(['cast', 'send', COUNTER, 'increment()',
     '--private-key', ALICE_PK, '--rpc-url', RPC])

$ cast send 0xe7f1725E7734CE288F8367e1Bb143E90bb3F0512 increment() --private-key 0xac0974bec39a17e36ba4a6b4d238ff944bacb478cbed5efcae784d7bf4f2ff80 --rpc-url http://127.0.0.1:8545



blockHash            0x284fd19e2c95f747e1f031d3bc01de6de5a693be1e252d67c3669c55265b80f0
blockNumber          3
contractAddress      
cumulativeGasUsed    43428
effectiveGasPrice    766607931
from                 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266
gasUsed              43428
logs                 []
logsBloom            0x00000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
root                 
status               1 (success)
transactionHash      0xdc9013c5555722961aa27a771469ee5fef578b71f5dd5537882f94e65d54c097
transactionInd

'blockHash            0x284fd19e2c95f747e1f031d3bc01de6de5a693be1e252d67c3669c55265b80f0\nblockNumber          3\ncontractAddress      \ncumulativeGasUsed    43428\neffectiveGasPrice    766607931\nfrom                 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266\ngasUsed              43428\nlogs                 []\nlogsBloom            0x00000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000\nroot                 \nstatus               1 (success)\ntransactionHash      0xdc9013c5555722961aa27a771469ee5fef578b71f5dd5537882f94e65d54c097\ntr

In [15]:
n = run(['cast', 'call', COUNTER, 'number()(uint256)', '--rpc-url', RPC])
print(f'\nnumber() = {n}')

$ cast call 0xe7f1725E7734CE288F8367e1Bb143E90bb3F0512 number()(uint256) --rpc-url http://127.0.0.1:8545


1

number() = 1


The receipt for the `increment` tx had a `gasUsed` field — that's the EVM measuring how much computational work the function did. Every opcode (ADD, SSTORE, etc.) has a fixed gas cost; the tx's gas budget must cover the total.

**Revert.** When a contract calls `require(...)` or `revert(...)` and the condition fails, all state changes from that tx are undone. The tx still gets mined and consumes gas, but its receipt has `status: 0`. Let's see it.

In [16]:
# Push number up to 5, then try a 6th increment which should revert.
for _ in range(4):
    run(['cast', 'send', COUNTER, 'incrementBounded()',
         '--private-key', ALICE_PK, '--rpc-url', RPC])
n = run(['cast', 'call', COUNTER, 'number()(uint256)', '--rpc-url', RPC])
print(f'\nnumber() = {n}  (expect 5)')

$ cast send 0xe7f1725E7734CE288F8367e1Bb143E90bb3F0512 incrementBounded() --private-key 0xac0974bec39a17e36ba4a6b4d238ff944bacb478cbed5efcae784d7bf4f2ff80 --rpc-url http://127.0.0.1:8545



blockHash            0x1758d4e44b12a8dc62a98b641bc77606829f998da848433745fcc0caf68d69bf
blockNumber          4
contractAddress      
cumulativeGasUsed    26406
effectiveGasPrice    671059376
from                 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266
gasUsed              26406
logs                 []
logsBloom            0x00000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
root                 
status               1 (success)
transactionHash      0x48c40f152ab5c4a7888785f25b44feb8c9b5361dee913c20d528803b432ab7e7
transactionInd


blockHash            0x60f67f768c5873d2c4b47c337e1d7d7c7d00227f0c572b6720c823991d295d2e
blockNumber          5
contractAddress      
cumulativeGasUsed    26406
effectiveGasPrice    587324621
from                 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266
gasUsed              26406
logs                 []
logsBloom            0x00000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
root                 
status               1 (success)
transactionHash      0x91f88289d0243c1c5558ef0f3f5e13b258f4678f5690b952dc9cfd5a8c617073
transactionInd


blockHash            0x21a6a4f98c0fabdfa485b43bb10f3a27fe175e1f97db26a582ce47728cf3ab69
blockNumber          6
contractAddress      
cumulativeGasUsed    26406
effectiveGasPrice    514038285
from                 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266
gasUsed              26406
logs                 []
logsBloom            0x00000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
root                 
status               1 (success)
transactionHash      0xb0e99e7f5e0a404db02957895c1af023ebdd9d56475b3f957907ba7df240f468
transactionInd

5

number() = 5  (expect 5)


In [17]:
# The 6th call should revert with "max reached".
# We pass check=False because we expect a non-zero exit.
run(['cast', 'send', COUNTER, 'incrementBounded()',
     '--private-key', ALICE_PK, '--rpc-url', RPC], check=False)

$ cast send 0xe7f1725E7734CE288F8367e1Bb143E90bb3F0512 incrementBounded() --private-key 0xac0974bec39a17e36ba4a6b4d238ff944bacb478cbed5efcae784d7bf4f2ff80 --rpc-url http://127.0.0.1:8545


Error: Failed to estimate gas: server returned an error response: error code 3: execution reverted: max reached, data: "0x08c379a00000000000000000000000000000000000000000000000000000000000000020000000000000000000000000000000000000000000000000000000000000000b6d61782072656163686564000000000000000000000000000000000000000000": Error("max reached")


''

In [18]:
# State unchanged — still 5.
n = run(['cast', 'call', COUNTER, 'number()(uint256)', '--rpc-url', RPC])
print(f'\nnumber() after revert = {n}  (still 5)')

$ cast call 0xe7f1725E7734CE288F8367e1Bb143E90bb3F0512 number()(uint256) --rpc-url http://127.0.0.1:8545


5

number() after revert = 5  (still 5)


Revert is the contract's safety hatch: any invalid condition unwinds the whole tx atomically. You'll see `BandwidthEscrow` use this extensively — every state-machine violation is a `revert WrongStatus(...)`.

## Teardown

Kill the anvil process. Re-run this notebook from the top to start fresh.

In [19]:
_anvil_proc.terminate()
_anvil_proc.wait(timeout=5)
print('anvil stopped')

anvil stopped
